# ARIF Phase 0 — エージェントペルソナ手動検証

**目的：** 5体のエージェントが3ラウンドの熟議を通じて「第三の解」を生成できるか、  
プロンプト品質を手動で検証・反復改善する。

> **「Phase 0 が最重要。プロンプトの質がシステム全体品質の 80% を決定する。」**  
> — 仕様書 §8

## 検証フロー
```
Round 1 → 各エージェント独立回答（多様性確認）
Round 2 → 反論義務プロンプト付き熟議（収束チェック）
Round 3 → 統合プロンプト → ThirdSolution（品質基準5条チェック）
Pre-mortem → 失敗原因5つ
```

## 合格基準
- 多様性スコア（Φ） ≥ 0.3
- 全員一致なし（≥ 1体が反論）
- ThirdSolution が品質基準5条を全て満たす
- Pre-mortem が5件生成される

---
## 0. セットアップ

In [ ]:
import os
import asyncio
import json
import textwrap
from itertools import combinations
from IPython.display import Markdown, display
import anthropic

# API キー（環境変数 ANTHROPIC_API_KEY から取得）
client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

# モデルバージョン（仕様書 §7 ピン留め）
MODEL_OPUS  = "claude-opus-4-20250514"
MODEL_HAIKU = "claude-haiku-4-5-20251001"

print(f"Anthropic SDK version: {anthropic.__version__}")
print(f"使用モデル（討論）: {MODEL_OPUS}")
print(f"使用モデル（要約）: {MODEL_HAIKU}")

---
## 1. テスト課題

> **Phase 0 テスト課題 Q1（テクノ中部 想定）**  
> テクノ中部の経営状況を踏まえた実戦的な問い。

In [ ]:
TEST_QUESTION = """
当社（テクノ中部）は2026年6月より、Garmin inReach を活用した遠隔地作業員安全管理システム
（StandbyCrew）を導入し、外販事業として展開を開始する。
国内市場（建設・エネルギー・林業）への深耕と、
米国・中東市場への海外展開の両方が視野に入っているが、
経営資源（人員・資金・時間）は限られている。

今後3年間で最大の成果を得るため、何を最優先すべきか。
国内先行 vs 海外先行 vs 並行展開、どの戦略が最善か。
また、この事業が長期的に競争優位を持ち続けるための条件は何か。
""".strip()

display(Markdown(f"### テスト課題\n\n{TEST_QUESTION}"))

---
## 2. エージェント定義

各エージェントのシステムプロンプトを定義する。  
**ここが Phase 0 の核心。プロンプトを反復改善して合格水準を目指す。**

In [ ]:
AGENTS = {
    "strategist": {
        "name": "経営戦略家",
        "temperature": 0.7,
        "system": """
あなたは30年のキャリアを持つ経営戦略の専門家です。
事業の5〜10年先を見通し、競合環境・業界構造・ポジショニングの観点から分析します。

あなたの役割：
- 長期的な事業方向性を鮮明に示すこと
- 「勝てる市場」と「勝てない市場」を峻別すること
- 競合が真似できない差別化軸を特定すること
- 大胆かつ具体的な戦略的選択肢を提示すること

回答スタイル：美辞麗句を避け、核心を突いた提言を。
「〜が重要です」という抽象論は禁止。「〜を先にやれ」「〜から撤退せよ」と断言せよ。
最後に「この戦略が機能する前提条件」を1〜2点明記すること。
""".strip()
    },
    "cfo": {
        "name": "CFO（財務・投資家）",
        "temperature": 0.3,
        "system": """
あなたは投資銀行出身のCFO（最高財務責任者）です。
ROI・キャッシュフロー・投資回収期間・単位経済性の観点から経営課題を分析します。

あなたの役割：
- 財務インパクトを定量的に示すこと（概算数値を必ず入れる）
- 投資対効果を厳格に評価すること
- 資金繰りリスクと損益分岐点を特定すること
- 「美しい戦略」より「数字が成立する戦略」を優先すること

回答スタイル：感情論・理念論は排除し、数字で語れ。
根拠のない楽観は厳禁。「この事業の単位経済性（Unit Economics）は〜」から始めること。
最後に「財務上の最大リスク」を1点明記すること。
""".strip()
    },
    "engineer": {
        "name": "技術・現場エンジニア",
        "temperature": 0.4,
        "system": """
あなたはベテランの現場エンジニア兼CTOです。
技術的実現可能性と現場の実態を誰よりも熟知しています。

あなたの役割：
- 技術的に実現可能かどうかを率直に評価すること
- 現場で必ず起きる問題を具体的に列挙すること
- 技術的リスクと現実的な対策を示すこと
- 「絵に描いた餅」の戦略を実装可能な設計に落とし込むこと

回答スタイル：「できる」「できない」を明確に。曖昧な楽観論は排除せよ。
Garmin inReach・IoT通信・SaaS運用の技術的観点を必ず含めること。
最後に「技術実装上の最大のボトルネック」を1点明記すること。
""".strip()
    },
    "market": {
        "name": "顧客・市場アナリスト",
        "temperature": 0.8,
        "system": """
あなたは顧客行動と市場動向の専門家です。
消費者心理・競合分析・市場トレンドの観点から分析します。

あなたの役割：
- 顧客が本当に求めているものを深く洞察すること
- 市場の潜在的な機会と脅威を特定すること
- 競合の動きと自社の差別化ポイントを明らかにすること
- 「作り手の論理」ではなく「買い手の論理」で考えること

回答スタイル：データ・事例・顧客の声（仮定でも可）を交えて語れ。
「市場規模は〜」「ターゲット顧客の最大の悩みは〜」を具体的に示すこと。
最後に「この市場で勝つための最重要な顧客インサイト」を1点明記すること。
""".strip()
    },
    "risk": {
        "name": "リスク・法務（悪魔の代弁者）",
        "temperature": 0.2,
        "system": """
あなたはリスク管理の専門家であり、この議論の「悪魔の代弁者」です。
最悪シナリオ・規制リスク・法務問題・実行上の落とし穴を徹底的に洗い出します。

あなたの役割：
- 他の意見の「危険な前提」を特定し、反論すること
- 誰も語りたくないリスクを明文化すること
- 規制・法令・コンプライアンス・保険・契約上の問題を指摘すること
- 「全員が賛成しているなら何かがおかしい」と疑うこと

回答スタイル：不人気でも構わない。正直に、辛口に。
「〜のリスクは軽視されている」「〜という前提は崩れる可能性がある」と断言せよ。
特に海外展開・法規制・事故発生時の責任問題に着目すること。
最後に「このビジネスモデルの致命的な急所」を1点明記すること。
""".strip()
    }
}

print(f"定義済みエージェント: {list(AGENTS.keys())}")
for agent_id, agent in AGENTS.items():
    print(f"  {agent_id}: {agent['name']} (temperature={agent['temperature']})")

---
## 3. Round 1 — 独立回答（相互参照なし）

In [ ]:
def call_agent(agent_id: str, question: str, extra_context: str = "") -> str:
    """1体のエージェントを呼び出す（同期版・Phase 0 検証用）"""
    agent = AGENTS[agent_id]
    user_content = question if not extra_context else f"{extra_context}\n\n---\n\n{question}"
    
    response = client.messages.create(
        model=MODEL_OPUS,
        max_tokens=2048,
        temperature=agent["temperature"],
        system=agent["system"],
        messages=[{"role": "user", "content": user_content}]
    )
    return response.content[0].text


# Round 1 実行（順次実行・Phase 0 は可読性優先）
print("Round 1 開始 — 5体が独立回答中...\n")
round1_responses = {}

for agent_id in AGENTS:
    print(f"  [{agent_id}] 回答生成中...", end=" ", flush=True)
    round1_responses[agent_id] = call_agent(agent_id, TEST_QUESTION)
    print("完了")

print("\nRound 1 完了")

In [ ]:
# Round 1 表示
for agent_id, response in round1_responses.items():
    agent_name = AGENTS[agent_id]["name"]
    display(Markdown(f"### [{agent_id}] {agent_name}\n\n{response}\n\n---"))

In [ ]:
# 多様性スコア計算（コサイン類似度ベース）
# Phase 0 では簡易版として単語重複率で代替

def simple_diversity_score(responses: dict) -> float:
    """簡易多様性スコア: 全ペアの単語重複率の逆数の平均"""
    texts = list(responses.values())
    pairs = list(combinations(range(len(texts)), 2))
    if not pairs:
        return 0.0
    
    overlap_scores = []
    for i, j in pairs:
        words_i = set(texts[i].split())
        words_j = set(texts[j].split())
        if not words_i or not words_j:
            continue
        overlap = len(words_i & words_j) / len(words_i | words_j)
        overlap_scores.append(overlap)
    
    avg_overlap = sum(overlap_scores) / len(overlap_scores)
    return round(1.0 - avg_overlap, 3)


r1_diversity = simple_diversity_score(round1_responses)
threshold = 0.3
status = "✅ 合格" if r1_diversity >= threshold else "❌ 要改善（プロンプト見直し）"

print(f"Round 1 多様性スコア: {r1_diversity}  （閾値: {threshold}）")
print(f"判定: {status}")

---
## 4. Round 2 — 反論・深掘り

全エージェントに「他者意見への反論義務」を付与する（仕様書 §4.2）。

In [ ]:
# Round 1 全意見の要約（Haikuで簡易要約）
all_r1_text = "\n\n".join(
    f"=== {AGENTS[aid]['name']} の意見 ===\n{resp}"
    for aid, resp in round1_responses.items()
)

summary_response = client.messages.create(
    model=MODEL_HAIKU,
    max_tokens=1500,
    messages=[{
        "role": "user",
        "content": f"以下の5つの意見を、各エージェントの主要論点が失われないよう1500字以内に要約せよ。\n\n{all_r1_text}"
    }]
)
round1_summary = summary_response.content[0].text

display(Markdown(f"### Round 1 要約（Haiku）\n\n{round1_summary}"))

In [ ]:
# Round 2 反論義務テンプレート（仕様書 §4.2）
ROUND2_TEMPLATE = """
## Round 2 の必須タスク

以下は他のエージェント5体による Round 1 の意見要約です：

{round1_summary}

---

上記を踏まえ、以下の3点に必ず答えよ：

1. **他のエージェントの意見で「最も危険な前提」を1つ特定し、具体的に反論せよ**
   （「危険な前提」とは：その前提が崩れた場合に戦略全体が機能しなくなるもの）

2. **自分の Round 1 意見で「最も弱い部分」を1つ自己批判せよ**
   （表面的な自己批判は不可。本当に脆い部分を指摘せよ）

3. **批判を踏まえ、修正した自分の立場を述べよ**
   （Round 1 と同じ結論でも構わないが、その場合は「なぜ変わらないか」を説明せよ）

※ もし全員が同じ結論に達していると感じたら、それ自体がリスクシグナルである。
  「なぜ全員が同じ方向を向いているのか」という構造的原因を分析すること。

---

元の課題：
{question}
""".strip()


# Round 2 実行
print("Round 2 開始 — 5体が反論・深掘り中...\n")
round2_responses = {}

r2_prompt = ROUND2_TEMPLATE.format(
    round1_summary=round1_summary,
    question=TEST_QUESTION
)

for agent_id in AGENTS:
    print(f"  [{agent_id}] Round 2 回答生成中...", end=" ", flush=True)
    round2_responses[agent_id] = call_agent(agent_id, r2_prompt)
    print("完了")

print("\nRound 2 完了")

In [ ]:
# Round 2 表示
for agent_id, response in round2_responses.items():
    agent_name = AGENTS[agent_id]["name"]
    display(Markdown(f"### [{agent_id}] {agent_name} — Round 2\n\n{response}\n\n---"))

In [ ]:
# 収束チェック（Haiku）
convergence_check = client.messages.create(
    model=MODEL_HAIKU,
    max_tokens=500,
    messages=[{
        "role": "user",
        "content": f"""以下の5つの Round 2 意見を読み、2点を判定せよ：
1. 全員が同一結論に達しているか（YES/NO + 理由）
2. 実質的な反論が発生しているか（YES/NO + 具体的にどのエージェントが誰に反論したか）

{chr(10).join(f'=== {AGENTS[aid]["name"]} ==={chr(10)}{resp}' for aid, resp in round2_responses.items())}"""
    }]
).content[0].text

display(Markdown(f"### 収束チェック\n\n{convergence_check}"))

---
## 5. Round 3 — 統合プロンプト → 第三の解

In [ ]:
# Round 2 要約（Haiku）
all_r2_text = "\n\n".join(
    f"=== {AGENTS[aid]['name']} の Round 2 意見 ===\n{resp}"
    for aid, resp in round2_responses.items()
)

round2_summary = client.messages.create(
    model=MODEL_HAIKU,
    max_tokens=1500,
    messages=[{
        "role": "user",
        "content": f"以下の Round 2 意見（反論・深掘りを含む）を1500字以内で要約せよ。各エージェントの最終的な立場の変化を明示せよ。\n\n{all_r2_text}"
    }]
).content[0].text

display(Markdown(f"### Round 2 要約（Haiku）\n\n{round2_summary}"))

In [ ]:
# Round 3 統合プロンプト（仕様書 §4.3・§4.4 準拠）
SYNTHESIS_PROMPT = """
あなたは5体の専門家エージェントによる3ラウンドの熟議を統合する「第三の解の起草者」です。

## 熟議の記録

### Round 1 要約（各エージェントの独立回答）
{round1_summary}

### Round 2 要約（反論・深掘り後）
{round2_summary}

## 元の課題
{question}

---

## 出力指示

以下の形式で「第三の解」を起草せよ。
「第三の解」とは、どのエージェントも提案しなかったが、複数の意見の干渉から生まれる新しい答えである。

### 品質基準（全て満たすこと）
1. **新規性**: Round 1 で誰も提案していない要素を含むこと
2. **統合性**: 少なくとも3つのエージェントの論点を取り込むこと（エージェント名を明示）
3. **実行可能性**: 具体的な次のアクションが明記されていること
4. **リスク認識**: 主要リスクとその緩和策が含まれていること
5. **時間軸**: 短期（3ヶ月以内）と中期（1〜3年）を区別すること

### 出力フォーマット（この順番で厳守）

■ 結論（断定形1文）
「〇〇すべきである。なぜなら〇〇と〇〇が干渉して第三の道〇〇が生まれたから。」

■ 論拠（エージェント名を明示して統合）

■ アクション【短期：3ヶ月以内】（3〜5項目）

■ アクション【中期：1〜3年】

■ 前提条件（これが崩れれば結論が変わる）

■ 少数意見（強く反対したエージェントの論点があれば記載）

■ 本提案は ARIF が熟議を経て導出した最善の一手です。最終判断は社長が下されるものです。
""".strip()


# Round 3 実行
print("Round 3 開始 — 第三の解を生成中...")

synthesis_text = client.messages.create(
    model=MODEL_OPUS,
    max_tokens=3000,
    temperature=0.5,
    messages=[{
        "role": "user",
        "content": SYNTHESIS_PROMPT.format(
            round1_summary=round1_summary,
            round2_summary=round2_summary,
            question=TEST_QUESTION
        )
    }]
).content[0].text

print("完了")
display(Markdown(f"## 第三の解\n\n{synthesis_text}"))

---
## 6. Pre-mortem（事前検死）

In [ ]:
pre_mortem_text = client.messages.create(
    model=MODEL_OPUS,
    max_tokens=1000,
    messages=[{
        "role": "user",
        "content": f"""以下の経営判断が1年後に大失敗していました。

{synthesis_text}

---

失敗の原因として最もありえるものを5つ、具体的かつ辛辣に列挙してください。
「可能性がある」ではなく「〜だったから失敗した」という断言形で書くこと。"""
    }]
).content[0].text

display(Markdown(f"## Pre-mortem（失敗原因5つ）\n\n{pre_mortem_text}"))

---
## 7. 品質評価チェックリスト

このセルは**手動で記入**する。出力を読んで各項目を評価せよ。

In [ ]:
# ============================================================
# 手動評価チェックリスト（1=不合格 / 2=要改善 / 3=合格）
# ============================================================

EVALUATION = {
    # Round 1 評価
    "r1_diversity_score": r1_diversity,          # 自動計算済み
    "r1_each_agent_distinctive": None,           # 各エージェントの視点が明確に異なるか（1/2/3）
    "r1_concrete_not_abstract": None,            # 抽象論でなく具体的提言か（1/2/3）

    # Round 2 評価
    "r2_real_counterargument": None,             # 実質的な反論があるか（1/2/3）
    "r2_no_full_convergence": None,              # 全員一致になっていないか（1/2/3）
    "r2_self_criticism_genuine": None,           # 自己批判が表面的でないか（1/2/3）

    # Round 3 評価（仕様書 §4.3 品質基準5条）
    "r3_novelty": None,                          # 新規性：R1に誰も提案していない要素を含むか（1/2/3）
    "r3_integration_3plus": None,                # 統合性：3体以上の論点を取り込んでいるか（1/2/3）
    "r3_actionable": None,                       # 実行可能性：具体的なアクションがあるか（1/2/3）
    "r3_risk_aware": None,                       # リスク認識：主要リスクと緩和策があるか（1/2/3）
    "r3_timeline_separated": None,               # 時間軸：短期と中期が区別されているか（1/2/3）

    # Pre-mortem 評価
    "pm_5_scenarios": None,                      # 5件生成されたか（1/2/3）
    "pm_specific_not_generic": None,             # 汎用的でなく本課題固有の失敗原因か（1/2/3）

    # 総合評価
    "overall_would_help_president": None,        # この出力は社長の意思決定に役立つか（1/2/3）
    "notes": ""                                  # 自由記述：プロンプト改善メモ
}

# ============================================================
# 以下に評価値を記入してください
# 例：
# EVALUATION["r1_each_agent_distinctive"] = 3    # 合格
# EVALUATION["r3_novelty"] = 2                    # 要改善
# EVALUATION["notes"] = "strategistが抽象的すぎる。数値を入れるよう修正する"
# ============================================================

print("評価チェックリストの各値を記入してから次のセルを実行してください。")
print(json.dumps({k: v for k, v in EVALUATION.items() if k != "notes"}, 
                  ensure_ascii=False, indent=2))

In [ ]:
# 評価集計・合否判定
scores = {k: v for k, v in EVALUATION.items() 
          if isinstance(v, (int, float)) and k != "r1_diversity_score"}

filled = {k: v for k, v in scores.items() if v is not None}
if filled:
    avg = sum(filled.values()) / len(filled)
    passed = sum(1 for v in filled.values() if v >= 3)
    needs_work = sum(1 for v in filled.values() if v == 2)
    failed = sum(1 for v in filled.values() if v == 1)

    print(f"=== Phase 0 評価サマリー ===")
    print(f"多様性スコア: {r1_diversity}  （閾値0.3: {'✅' if r1_diversity >= 0.3 else '❌'}）")
    print(f"平均スコア: {avg:.2f} / 3.0")
    print(f"合格（3）: {passed}項目 / 要改善（2）: {needs_work}項目 / 不合格（1）: {failed}項目")
    print()

    if avg >= 2.5 and failed == 0:
        print("✅ Phase 0 合格。プロンプトを engine/agents.py に移植してよい。")
    elif avg >= 2.0:
        print("⚠️ 要改善。評価2の項目のプロンプトを修正して再実行せよ。")
    else:
        print("❌ 不合格。エージェント定義を大幅に見直して再実行せよ。")

    if EVALUATION.get("notes"):
        print(f"\n改善メモ: {EVALUATION['notes']}")
else:
    print("⚠️ 評価値が未記入です。上のセルで各項目に 1/2/3 を記入してください。")

---
## 8. 反復改善ガイド

| 問題 | 対処法 |
|------|--------|
| Round 1 の多様性が低い | 各エージェントのシステムプロンプトに「他との差別化」を明示。温度パラメータを広げる |
| Round 2 で全員一致 | risk エージェントのプロンプトに反論強制を追加。R2テンプレートの設問を鋭くする |
| 第三の解が抽象的 | SYNTHESIS_PROMPT の品質基準に「具体的な社名・数値・期日を入れよ」を追加 |
| Pre-mortem が汎用的 | Pre-mortem プロンプトに「この業界・この規模・この国特有の失敗原因」を要求 |
| 特定エージェントが弱い | そのエージェントのシステムプロンプトのみ単独で呼び出してチューニングする |

## Phase 0 完了条件

- [ ] 評価スコア平均 2.5 以上・不合格なし
- [ ] 多様性スコア Φ ≥ 0.3
- [ ] 第三の解が「社長がすぐに動ける」レベルの具体性を持つ
- [ ] 5体のシステムプロンプトを `engine/agents.py` に移植済み
- [ ] 仕様書 §9 未決事項「Phase 0 でのペルソナ検証結果」をチェック